# Week 08 — Home exercise 5: A merge that goes wrong

**Solution proposal.**

Diagnosing a join that runs perfectly and quietly deletes 48 rows — the lecture's diagnosis, done
again from scratch — and then the question the lecture did not ask: what in the lookup file matched
nothing at all?

This notebook loads the emissions file as it comes, with all 6 240 rows, so its counts differ from
the lecture's, which dropped the rows with no emissions figure first. The 48 are the same 48.

In [1]:
import pandas as pd

co2 = pd.read_csv("../data/co2_emissions.csv")
info = pd.read_csv("../data/country_info.csv")

print("emissions:", co2.shape)
print("lookup:   ", info.shape)

emissions: (6240, 11)
lookup:    (295, 4)


## 1. The inner join

In [2]:
inner = co2.merge(info, left_on="country", right_on="name", how="inner")

print("rows before:", len(co2))
print("rows after: ", len(inner))
print("lost:       ", len(co2) - len(inner))

rows before: 6240
rows after:  6192
lost:        48


**48 rows gone**, and no error, no warning, nothing in the output of the merge itself that says so.
The only reason we know is that we counted.

48 out of 6 240 is 0.8%. It would survive a `.head()`, a `.describe()`, a plot, and a conversation
with your supervisor.

## 2. The same merge as a left join, with `indicator=True`

In [3]:
checked = co2.merge(
    info,
    left_on="country",
    right_on="name",
    how="left",
    indicator=True,
)

print("rows:", len(checked))
checked["_merge"].value_counts()

rows: 6240


_merge
both          6192
left_only       48
right_only       0
Name: count, dtype: int64

Nothing is deleted now, and the `_merge` column says what happened to each row: **6 192 matched, 48
did not**, and with a left join nothing can come from the right side alone.

The 48 add up exactly with the inner join's loss, which confirms the diagnosis rather than assuming
it.

## 3. Which entities, and why

In [4]:
unmatched = checked[checked["_merge"] == "left_only"]["country"].unique()

print(list(unmatched))
print("years each:", len(checked[checked["country"] == unmatched[0]]))

['Latin America & Caribbean', 'Sub-Saharan Africa']
years each: 24


Two entities, 24 years apiece. And they are obviously in the lookup file — you can see them.

In [5]:
print("present in the lookup file?")
for name in unmatched:
    matches = (info["name"].str.strip() == name).sum()
    print(" ", name, "->", matches, "match after stripping")

present in the lookup file?
  Latin America & Caribbean -> 1 match after stripping
  Sub-Saharan Africa -> 1 match after stripping


So the entity **is** there, and the merge still failed. That rules out "missing data" and points at
the key itself. Printing the lookup file's versions as a **list** shows each string in quotes, so the
edges become visible — and their lengths settle it.

In [6]:
suspects = info[info["name"].str.strip().isin(unmatched)]["name"]

print(suspects.tolist())
print("lengths:         ", suspects.str.len().tolist())
print("lengths stripped:", suspects.str.strip().str.len().tolist())

['Latin America & Caribbean ', 'Sub-Saharan Africa ']
lengths:          [26, 19]
lengths stripped: [25, 18]


**A single trailing space.** `'Sub-Saharan Africa '` is 19 characters and `'Sub-Saharan Africa'` is
18, and no table pandas displays will ever show you the difference.

This is exactly how the World Bank publishes the file, so it is not a corruption to be reported —
it is a property of real data to be handled.

## 4. Two fixes

**Fix A: clean the key.** `.str.strip()` on both sides, so that whatever whitespace either file
carries is gone before the comparison.

In [7]:
co2_clean = co2.copy()
info_clean = info.copy()

co2_clean["country"] = co2_clean["country"].str.strip()
info_clean["name"] = info_clean["name"].str.strip()

fix_a = co2_clean.merge(info_clean, left_on="country", right_on="name", how="left")

print("rows:", len(fix_a), "| region missing:", fix_a["region"].isna().sum())

rows: 6240 | region missing: 0


**Fix B: use a key that was never dirty.** Both files carry the ISO three-letter `code`, which has no
whitespace, no capitalization question and no spelling variants.

In [8]:
fix_b = co2.merge(info[["code", "region", "incomeLevel"]], on="code", how="left")

print("rows:", len(fix_b), "| region missing:", fix_b["region"].isna().sum())

rows: 6240 | region missing: 0


Both give 6 240 rows with nothing missing.

### Which would I use?

**Here, `code`** — and it is not close. A code is designed to be a key: it is short, it has one
canonical form, and it cannot be misspelled without becoming obviously invalid. A country name is
designed to be read by humans, and it varies by whitespace, by capitalization, by punctuation
(`Cote d'Ivoire`), by language, and by politics (`Korea, Rep.` against `South Korea`).

**On a file with no code column**, you have no choice, and then `.str.strip()` on both sides before
merging is the minimum. Realistically you would also `.str.lower()` for the comparison, and then check
what still failed to match with `indicator=True` — because whitespace is the *easiest* of the problems
a name key has, and fixing it just reveals the next one.

The general rule: **merge on the most machine-made column available.** Prefer a code to a name, an id
to a code, and a name only when nothing else exists.

## 5. `validate=`

In [9]:
panel = co2.merge(
    info[["code", "region", "incomeLevel"]],
    on="code",
    how="left",
    validate="many_to_one",
)

print("validated, rows:", len(panel))

validated, rows: 6240


It held. Now claim the wrong shape on purpose.

In [10]:
# MergeError
co2.merge(info[["code", "region"]], on="code", how="left", validate="one_to_one")

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
 code
 AFG
 AFG
 AFG
 AFG
 AFG ...

`one_to_one` is false because the emissions file has 24 rows per country — which is true and obvious,
and is precisely the kind of thing people get wrong about data they did not create.

The value of `validate=` is that it turns a **belief** into a **check**. Writing `many_to_one` is
saying out loud "I think this lookup has one row per country", and if a future version of the file
gains a duplicate, the merge stops instead of silently doubling your rows.

## The other side of the merge

A left join cannot show you lookup entries that matched nothing, because it never keeps them. An
**outer** join keeps unmatched rows from both sides, and `indicator=True` labels them `right_only`.

In [11]:
both_sides = co2.merge(info, on="code", how="outer", indicator=True)

print(both_sides["_merge"].value_counts())

_merge
both          6240
right_only      35
left_only        0
Name: count, dtype: int64


In [12]:
unused = both_sides[both_sides["_merge"] == "right_only"]

print("unused lookup entries:", len(unused))
print(unused["region"].value_counts())
print()
print(unused["name"].head(8).tolist())

unused lookup entries: 35
region
Aggregates    35
Name: count, dtype: int64

['Africa', 'East Asia & Pacific (IBRD-only countries)', 'Europe & Central Asia (IBRD-only countries)', 'IBRD countries classified as high income', 'Latin America & the Caribbean (IBRD-only countries)', 'Middle East, North Africa, Afghanistan & Pakistan (IBRD only)', 'Sub-Saharan Africa (IBRD-only countries)', 'Sub-Saharan Africa (IFC classification)']


**35 of the 295 lookup entries match nothing, and every one of them is an aggregate** — more World
Bank groupings, such as the IBRD-only and IDA-eligible versions of each region, and the four income
groups `High income`, `Low income`, `Lower middle income` and `Upper middle income`. The lookup file
lists every grouping the bank uses; the emissions file was downloaded with only some of them.

**For this analysis they do not matter.** The lookup is a decoration table: we bring its columns across
to the emissions rows, and an entry with no emissions row has nowhere to go. And since we drop the
aggregates anyway, these are entities we would have removed regardless.

**Unused lookup entries matter when the lookup is supposed to be complete in the other direction**
— when it is the list of what *should* be in your data. If the lookup were "every country in the
survey" and three countries came back `right_only`, those would be three countries whose data never
arrived, and nothing else in the analysis would ever tell you. Four of the 35 here carry a milder
warning of that kind: the income groups are in the lookup and not in the emissions file, so any
question about income groups has to be answered by adding up countries, because the bank's own
totals for them are not here.

### Things worth noticing

- **The diagnosis had four steps and only the last one was the fix.** Count, then `indicator=`, then
  find the rows, then look at the key itself. Jumping straight to "use `code` instead" gets the right
  answer this time and teaches you nothing about the next file.
- **Printing as a list is the tool for invisible characters.** Trailing spaces, tabs and non-breaking
  spaces all look like nothing at all in a table, until you see the quotes around the string.
- **A left join plus `indicator=True` is non-destructive**, and an outer join plus `indicator=True`
  shows both directions at once. Either is the right first move whenever a merge surprises you,
  because it lets you look at the failures instead of at their absence.

### What this notebook does NOT do

- `.str.strip()` handles whitespace at the ends and nothing else. `"Cote d'Ivoire"` against
  `"Côte d'Ivoire"`, or `"Korea, Rep."` against `"South Korea"`, would still fail, and no amount of
  stripping helps — those need a lookup table of alternative names, which is a real and tedious part
  of this kind of work.
- Dropping every entity whose region is `"Aggregates"` trusts one column of one file completely. It is
  right here, and it is worth knowing that the whole result rests on it.